Purpose: Investigate genic overlap across genotypes (overall & by physiology) within each ZTpeak cluster group.<br>
Author: Anna Pardo<br>
Date initiated: Feb. 27, 2026

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

In [2]:
# load time-structured gene information
cinfo = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/masigpro_results/clusters_ngenes_gt_ztgroup_with36-61-43.txt",sep="\t",header="infer")
cinfo.head()

,cluster,genotype,n_CAM_genes,n_all_genes,gtc,ZTpeak
0,1,53,0.0,319,53_1,5.0
1,2,53,1.0,656,53_2,21.0
2,3,53,1.0,882,53_3,17.0
3,4,53,0.0,1071,53_4,9.0
4,5,53,0.0,428,53_5,21.0


In [3]:
# load CAM gene annotation
cam = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/degs_downstream/camgenes_Ya_Yf_orthology_synteny.txt",sep="\t",header="infer")

In [4]:
# load physiology annotation
phys = json.load(open("/home/leviathan22/Yucca_genomics/phys_figures/physiological_CAM_categories.json"))

In [9]:
# load cluster membership
def load_sum_gt(gt,d):
    if gt in ["all","gt37"]:
        if gt=="all":
            gtc = pd.read_csv(os.path.join(d,"hck6_treatonly_"+gt+"_18-Nov.tsv"),sep="\t",header="infer")
        else:
            gtc = pd.read_csv(os.path.join(d,"hck6_treatonly_"+gt+"_17-Nov.tsv"),sep="\t",header="infer")
    else:
        gtc = pd.read_csv(os.path.join(d,"Yg"+gt+"_hclust_k6_clusters.txt"),sep="\t",header="infer")
    sum1 = gtc.groupby("cluster").count().reset_index().rename(columns={"GeneID":"n_genes_"+gt})
    gtcam = cam.merge(gtc)
    camsum = gtcam.groupby("cluster").count().reset_index()[["cluster","GeneID"]].rename(columns={"GeneID":"n_genes_"+gt})
    d = {"clusters":gtc,"CAMgenes":gtcam,"all_summary":sum1,"CAM_summary":camsum}
    return d

In [11]:
allgt = {}
d="/home/leviathan22/Yucca_genomics/rna_insilico_genome/masigpro_results/"
for f in os.listdir(d):
    if f.startswith("Yg"):
        g = f.split("_")[0].lstrip("Yg")
        allgt[g] = load_sum_gt(g,d)
    elif f.startswith("hck6"):
        g = f.split("_")[2]
        df = load_sum_gt(g,d)
        if g=="gt37":
            g = g.lstrip("gt")
        allgt[g] = df

In [14]:
allgt["18"]["clusters"].head()

,cluster,GeneID
0,1,Yucal.01G000100.v2.1
1,2,Yucal.01G000900.v2.1
2,3,Yucal.01G001000.v2.1
3,4,Yucal.01G001200.v2.1
4,4,Yucal.01G001300.v2.1


In [15]:
dflist = []
for k,v in allgt.items():
    df = v["clusters"]
    df["genotype"] = k
    sg = []
    for i in list(df["GeneID"]):
        if i.startswith("Yucal"):
            sg.append("Ya")
        else:
            sg.append("Yf")
    df["subgenome"] = sg
    dflist.append(df)

In [28]:
all_cluster_mem = pd.concat(dflist)

In [29]:
all_cluster_mem = all_cluster_mem[all_cluster_mem["genotype"]!="all"]

In [30]:
all_cluster_mem["CAMphys"] = all_cluster_mem["genotype"].map(phys)
all_cluster_mem.head()

/tmp/ipykernel_31377/609171646.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_cluster_mem["CAMphys"] = all_cluster_mem["genotype"].map(phys)


,cluster,GeneID,genotype,subgenome,CAMphys
0,1,Yucal.01G003400.v2.1,G,Ya,C3
1,1,Yucal.01G004800.v2.1,G,Ya,C3
2,2,Yucal.01G004900.v2.1,G,Ya,C3
3,3,Yucal.01G005300.v2.1,G,Ya,C3
4,4,Yucal.01G012100.v2.1,G,Ya,C3


In [31]:
all_cluster_mem["gtc"] = all_cluster_mem["genotype"]+"_"+all_cluster_mem["cluster"].astype(str)

/tmp/ipykernel_31377/122836736.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_cluster_mem["gtc"] = all_cluster_mem["genotype"]+"_"+all_cluster_mem["cluster"].astype(str)


In [32]:
cinfo["CAMphys"] = cinfo["genotype"].map(phys)
cinfo.head()

,cluster,genotype,n_CAM_genes,n_all_genes,gtc,ZTpeak,CAMphys
0,1,53,0.0,319,53_1,5.0,possible_fac_CAM
1,2,53,1.0,656,53_2,21.0,possible_fac_CAM
2,3,53,1.0,882,53_3,17.0,possible_fac_CAM
3,4,53,0.0,1071,53_4,9.0,possible_fac_CAM
4,5,53,0.0,428,53_5,21.0,possible_fac_CAM


In [22]:
# key question of this notebook: for each ZTpeak, what (percentage of) genes overlap across each genotype set?
## where genotype sets = all genotypes and each CAMphys set

# set up a dict of key=gtc, value=ZTpeak
ztd = {}
for i in range(len(cinfo.index)):
    ztd[cinfo.iloc[i,4]] = cinfo.iloc[i,5]

In [33]:
all_cluster_mem["genotype"].unique()

array(['G', 'Eudy', '36', '43', '13', '45', '19', '51', '2AB', '52', '46',
       '70', '48', '37', '55', '18', '61', '53', '56', '1AB'],
      dtype=object)

In [34]:
all_cluster_mem["ZTpeak"] = all_cluster_mem["gtc"].map(ztd)
all_cluster_mem.head()

/tmp/ipykernel_31377/1641432204.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_cluster_mem["ZTpeak"] = all_cluster_mem["gtc"].map(ztd)


,cluster,GeneID,genotype,subgenome,CAMphys,gtc,ZTpeak
0,1,Yucal.01G003400.v2.1,G,Ya,C3,G_1,1.0
1,1,Yucal.01G004800.v2.1,G,Ya,C3,G_1,1.0
2,2,Yucal.01G004900.v2.1,G,Ya,C3,G_2,9.0
3,3,Yucal.01G005300.v2.1,G,Ya,C3,G_3,5.0
4,4,Yucal.01G012100.v2.1,G,Ya,C3,G_4,21.0


In [36]:
def find_zt_gtint(gtlist,bias,df=all_cluster_mem):
    if bias!="both":
        bdf = df[df["subgenome"]==bias]
    else:
        bdf = df.copy()
    
    gsets = []
    for g in gtlist:
        gtdf = bdf[bdf["genotype"]==g]
        gsets.append(set(list(gtdf["GeneID"])))
    
    inter = list(set.intersection(*gsets))
    return inter

In [42]:
all_cluster_mem.head()

,cluster,GeneID,genotype,subgenome,CAMphys,gtc,ZTpeak
0,1,Yucal.01G003400.v2.1,G,Ya,C3,G_1,1.0
1,1,Yucal.01G004800.v2.1,G,Ya,C3,G_1,1.0
2,2,Yucal.01G004900.v2.1,G,Ya,C3,G_2,9.0
3,3,Yucal.01G005300.v2.1,G,Ya,C3,G_3,5.0
4,4,Yucal.01G012100.v2.1,G,Ya,C3,G_4,21.0


In [44]:
intersects = {}
for x in all_cluster_mem["ZTpeak"].unique():
    df = all_cluster_mem[all_cluster_mem["ZTpeak"]==x]
    intersects[x] = {}
    for i in ["Ya","Yf","both"]:
        intersects[x][i] = {}
        for j in ["all"]+list(all_cluster_mem["CAMphys"].unique()):
            if j=="all":
                intersects[x][i][j] = find_zt_gtint(list(all_cluster_mem["genotype"].unique()),i,df)
            else:
                gl = list(df[df["CAMphys"]==j]["genotype"].unique())
                intersects[x][i][j] = find_zt_gtint(gl,i,df)

In [45]:
# find the lengths of the intersects & compare with the numbers of genes overall in the cluster sets
lendict = {"ZTpeak":[],"subgenome":[],"genotype_set":[],"n_intersect_genes":[]}
for k,v in intersects.items():
    for i,j in v.items():
        for x,y in j.items():
            lendict['ZTpeak'].append(k)
            lendict['genotype_set'].append(x)
            lendict['subgenome'].append(i)
            lendict['n_intersect_genes'].append(len(y))
interlen = pd.DataFrame(lendict)

In [49]:
cct = all_cluster_mem.groupby(["ZTpeak","subgenome","CAMphys"]).count().reset_index()[["ZTpeak","subgenome","CAMphys","cluster"]]

In [50]:
allct = all_cluster_mem.groupby(["ZTpeak","subgenome"]).count().reset_index()
allct["CAMphys"] = "all"
allct = allct[["ZTpeak","subgenome","CAMphys","cluster"]]

counts = pd.concat([cct,allct])

In [51]:
counts.head()

,ZTpeak,subgenome,CAMphys,cluster
0,1.0,Ya,C3,13374
1,1.0,Ya,facultative_CAM,5155
2,1.0,Ya,possible_fac_CAM,19944
3,1.0,Yf,C3,12723
4,1.0,Yf,facultative_CAM,4989


In [52]:
interlen.head()

,ZTpeak,subgenome,genotype_set,n_intersect_genes
0,1.0,Ya,all,0
1,1.0,Ya,C3,5
2,1.0,Ya,facultative_CAM,100
3,1.0,Ya,possible_fac_CAM,1
4,1.0,Yf,all,0


In [53]:
interlen.merge(counts.rename(columns={"CAMphys":"genotype_set","cluster":"n_total_genes"}))

,ZTpeak,subgenome,genotype_set,n_intersect_genes,n_total_genes
0,1.0,Ya,all,0,38473
1,1.0,Ya,C3,5,13374
2,1.0,Ya,facultative_CAM,100,5155
3,1.0,Ya,possible_fac_CAM,1,19944
4,1.0,Yf,all,0,36732
5,1.0,Yf,C3,1,12723
6,1.0,Yf,facultative_CAM,109,4989
7,1.0,Yf,possible_fac_CAM,1,19020
8,9.0,Ya,all,0,42989
9,9.0,Ya,C3,0,15765
